# Single-Pass Extract Agent — Interactive Testing

Run the **exact same** `build_extract_agent` + `run_extraction` code path as production.
Same prompt, same tools (7), same history processors (3), same auto-scaled budget.

Change `TRACE_FILE` and `USE_MODEL` below to test different traces and models.

In [ ]:
import sys, os, time, json, tempfile
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

# sys.path.insert(0, str(Path("../src").resolve()))
# sys.path.insert(0, str(Path("../../lerim-cloud").resolve()))

# ── TRACE FILE ──────────────────────────────────────────────────────
TRACE_FILE = "claude_009"  # just the stem, e.g. "claude_003", "claude_013", "claude_028", "claude_009", "claude_007"
TRACE = Path(f"/Users/kargarisaac/codes/personal/lerim/lerim-cloud/evals/data/traces/{TRACE_FILE}.jsonl")
assert TRACE.exists(), f"Trace not found: {TRACE}"

# ── MODEL CONFIG ────────────────────────────────────────────────────
# Switch USE_MODEL to test different providers/models.
# All use the same build_pydantic_model_from_provider → same agent.
USE_MODEL = "minimax-m2.7"

MODEL_CONFIGS = {
    "minimax-m2.7": {
        "provider": "minimax",
        "model": "MiniMax-M2.7",
        "fallbacks": [],
    },
    "glm-5.1": {
        "provider": "zai",
        "model": "glm-5.1",
        "fallbacks": [],
    },
    "ollama-gpt-oss-20b": {
        "provider": "ollama",
        "model": "gpt-oss:20b",
        "fallbacks": [],
    },
    "ollama-qwen3-8b": {
        "provider": "ollama",
        "model": "qwen3:8b",
        "fallbacks": [],
    },
    "ollama-qwen3-30b": {
        "provider": "ollama",
        "model": "qwen3:30b-a3b",
        "fallbacks": [],
    },
}
MODEL_CFG = MODEL_CONFIGS[USE_MODEL]

from lerim.agents.tools import compute_request_budget

trace_lines = sum(1 for _ in open(TRACE))
budget = compute_request_budget(TRACE)
print(f"Trace:  {TRACE.name} ({trace_lines} lines)")
print(f"Budget: {budget} requests (auto-scaled)")
print(f"Model:  {USE_MODEL} → {MODEL_CFG['provider']}/{MODEL_CFG['model']}")

Trace:  claude_009.jsonl (363 lines)
Budget: 50 requests (auto-scaled)
Model:  minimax-m2.7 → minimax/MiniMax-M2.7


In [12]:
def make_temp_memory():
    """Create a fresh temp memory directory."""
    tmp = Path(tempfile.mkdtemp(prefix="nb_extract_"))
    mem = tmp / "memory"
    mem.mkdir(parents=True)
    (mem / "summaries").mkdir()
    (mem / "index.md").write_text("# Memory Index\n")
    return tmp, mem


def report(mem: Path):
    """Print what the agent wrote."""
    import frontmatter as fm_lib
    memories = [f for f in sorted(mem.glob("*.md")) if f.name != "index.md"]
    summaries = list(sorted((mem / "summaries").glob("*.md")))
    print(f"\nMemories written: {len(memories)}")
    for m in memories:
        try:
            post = fm_lib.load(str(m))
            print(f"  [{post.get('type','')}] {m.name}")
            print(f"    desc: {post.get('description','')[:120]}")
            print(f"    body: {post.content.strip()[:200]}")
        except Exception as e:
            print(f"  [YAML ERROR] {m.name} ({type(e).__name__})")
            print(f"    {m.read_text()[:200]}")
    print(f"\nSummaries written: {len(summaries)}")
    for s in summaries:
        print(f"  {s.name}")
        print(f"    {s.read_text()[:300]}")
    print(f"\nIndex:\n{(mem / 'index.md').read_text()}")

## Run Extraction

Uses `build_extract_agent` (7 tools + 3 history processors) and `run_extraction` with auto-scaled budget — identical to the production code path in `lerim.agents.extract`.

In [13]:
from lerim.agents.extract import build_extract_agent, SYSTEM_PROMPT
from lerim.agents.tools import ExtractDeps, compute_request_budget
from lerim.config.providers import build_pydantic_model_from_provider
from pydantic_ai.usage import UsageLimits

# Build model — same provider builder as production (HTTP retry + fallback)
model = build_pydantic_model_from_provider(
    MODEL_CFG["provider"],
    MODEL_CFG["model"],
    fallback_models=MODEL_CFG.get("fallbacks"),
)

# Build agent — exact same function as production
agent = build_extract_agent(model)

# Fresh memory directory
tmp_dir, mem_dir = make_temp_memory()

# Deps + budget — exact same as run_extraction()
deps = ExtractDeps(memory_root=mem_dir, trace_path=TRACE)
usage_limits = UsageLimits(request_limit=budget)

# print(f"Agent: {len(agent._function_tools)} tools, retries={agent._max_retries}")
# print(f"Memory root: {mem_dir}")
# print(f"Running...")

t0 = time.time()
result = await agent.run(
    "Extract memories from the session trace. Follow the core cycle: "
    "read → note → prune → synthesize → write.",
    deps=deps,
    usage_limits=usage_limits,
)
elapsed = time.time() - t0

print(f"\nDone in {elapsed:.1f}s")
print(f"Result: {result.output.completion_summary}")
print(f"Messages: {len(result.all_messages())}")
print(f"Notes collected: {len(deps.notes)}")
print(f"Pruned offsets: {deps.pruned_offsets or 'none'}")
report(mem_dir)


Done in 388.3s
Result: Wrote 1 summary (Per-Project Memory Isolation Part B) + 1 project memory (unmatched sessions silently skipped) + 1 feedback memory (worktree vs main repo path). Fixed index.md paths from subdirectory-relative to root-relative format. Verified index OK.
Messages: 45
Notes collected: 6
Pruned offsets: none

Memories written: 2
  [feedback] feedback_worktree_cwd_vs_main_repo_path.md
    desc: Running inside a worktree requires using worktree paths and reinstalling from the worktree
    body: When running inside a git worktree, always use worktree-relative file paths for edits — not the main repo absolute path.
**Why:** The `Read` tool uses absolute paths directly, bypassing cwd. The agent
  [project] project_unmatched_sessions_silently_skipped.md
    desc: Sessions with no repo_path or no project match are skipped (not queued for extraction)
    body: Sessions without a matching registered project are silently skipped during daemon extraction (completed with `"stat

## Tool Call Analysis

Classify each tool call as clean, XML-bleed (MiniMax native format leak), or empty-args. Shows per-tool breakdown and the ordered tool sequence.

In [14]:
messages = list(result.all_messages())

# ── Classify tool calls ─────────────────────────────────────────────
xml_bleed = 0
empty_args = 0
clean = 0
total = 0
tool_clean = Counter()
tool_bad = Counter()
tool_sequence = []  # (tool_name, args_preview, is_clean)

for msg in messages:
    for part in msg.parts:
        if part.part_kind != "tool-call":
            continue
        total += 1
        name = part.tool_name
        args = part.args
        args_str = json.dumps(args) if isinstance(args, dict) else str(args)

        is_xml = "<arg_key>" in args_str or "<arg_value>" in args_str or "<tool_call>" in args_str
        is_empty = False
        if not is_xml:
            try:
                args_dict = json.loads(args_str) if isinstance(args, str) else args
            except (json.JSONDecodeError, TypeError):
                args_dict = {}
            is_empty = (
                name not in ("verify_index", "final_result")
                and (args_dict == {}
                     or (name == "read" and not args_dict.get("filename", "").strip() and "offset" not in args_dict)
                     or (name == "write" and not args_dict.get("type", "").strip())
                     or (name == "note" and not args_dict.get("findings")))
            )

        if is_xml:
            xml_bleed += 1
            tool_bad[name] += 1
            tag = "XML"
        elif is_empty:
            empty_args += 1
            tool_bad[name] += 1
            tag = "EMPTY"
        else:
            clean += 1
            tool_clean[name] += 1
            tag = "OK"
        tool_sequence.append((name, args_str[:100], tag))

# ── Summary ──────────────────────────────────────────────────────────
print(f"Total tool calls: {total}")
print(f"  Clean:     {clean} ({clean*100//max(total,1)}%)")
print(f"  XML-bleed: {xml_bleed} ({xml_bleed*100//max(total,1)}%)")
print(f"  Empty:     {empty_args} ({empty_args*100//max(total,1)}%)")
print()

print("Per-tool breakdown (clean / bad):")
all_tools = sorted(set(list(tool_clean.keys()) + list(tool_bad.keys())))
for t in all_tools:
    c = tool_clean.get(t, 0)
    b = tool_bad.get(t, 0)
    pct = c * 100 // max(c + b, 1)
    print(f"  {t:20s}: {c:2d} clean, {b:2d} bad  ({pct}% clean)")
print()

print("Tool sequence:")
for i, (name, args_preview, tag) in enumerate(tool_sequence, 1):
    marker = "  " if tag == "OK" else f"⚠{tag}"
    print(f"  T{i:02d} [{marker:>5s}] {name}({args_preview})")

Total tool calls: 24
  Clean:     24 (100%)
  XML-bleed: 0 (0%)
  Empty:     0 (0%)

Per-tool breakdown (clean / bad):
  Bash                :  1 clean,  0 bad  (100% clean)
  edit                :  3 clean,  0 bad  (100% clean)
  final_result        :  1 clean,  0 bad  (100% clean)
  grep                :  1 clean,  0 bad  (100% clean)
  note                :  1 clean,  0 bad  (100% clean)
  read                : 10 clean,  0 bad  (100% clean)
  verify_index        :  4 clean,  0 bad  (100% clean)
  write               :  3 clean,  0 bad  (100% clean)

Tool sequence:
  T01 [     ] read({"filename": "index.md"})
  T02 [     ] read({"filename": "trace", "limit": 100, "offset": 0})
  T03 [     ] grep({"filename": "trace", "pattern": "remember|memorize|keep in mind|note this"})
  T04 [     ] read({"filename": "trace", "limit": 100, "offset": 100})
  T05 [     ] read({"filename": "trace", "limit": 100, "offset": 200})
  T06 [     ] read({"filename": "trace", "limit": 100, "offset": 294})
 

In [ ]:
# ── Full conversation log ────────────────────────────────────────────
print(f"{'=' * 70}")
print(f"CONVERSATION LOG ({len(messages)} messages)")
print(f"{'=' * 70}")

for i, msg in enumerate(messages):
    kind = msg.kind
    print(f"\n--- Message {i} [{kind}] ---")
    for part in msg.parts:
        ptype = part.part_kind
        if ptype == "system-prompt":
            print(f"  [system-prompt] ({len(part.content)} chars)")
        elif ptype == "user-prompt":
            print(f"  [user-prompt] {part.content}")
        elif ptype == "tool-return":
            content_str = str(part.content)
            print(f"  [tool-return] {part.tool_name} -> {content_str}")
        elif ptype == "text":
            print(f"  [text] {part.content}")
        elif ptype == "tool-call":
            args_str = json.dumps(part.args) if isinstance(part.args, dict) else str(part.args)
            print(f"  [tool-call] {part.tool_name}({args_str})")
        else:
            print(f"  [{ptype}] {str(part)}")

CONVERSATION LOG (45 messages)

--- Message 0 [request] ---
  [system-prompt] (8386 chars)
  [user-prompt] Extract memories from the session trace. Follow the core cycle: read → note → prune → synthesize → write.
  [system-prompt] (27 chars)
  [system-prompt] (42 chars)

--- Message 1 [response] ---
  [thinking] ThinkingPart(content='Let me start by following the core flow. I need to:\n1. ORIENT: read index.md, verify_index, grep for "remember|memorize|keep in mind|note this"\n2. SCAN: read trace in chunks, n
  [text] 

I'll follow the core extraction cycle. Starting with ORIENT.

  [tool-call] read({"filename": "index.md"})
  [tool-call] read({"filename": "trace", "limit": 100, "offset": 0})
  [tool-call] grep({"filename": "trace", "pattern": "remember|memorize|keep in mind|note this"})

--- Message 2 [request] ---
  [tool-return] read -> 1	# Memory Index
  [tool-return] read -> [363 lines, showing 1-100] — 263 more lines, call read('trace', offset=100, limit=100) for the next chunk
1